### Imports & Configuration

In [2]:
#!/usr/bin/env python3
from utils.logger import BaseLogger
from utils.dependencies import *
from utils.data_process import _dtypes_convert
from utils.table_reader import *
from utils.queries import pollaploi_query, parohes_cosmote_query 
from utils.type_mapping import type_mapping
from utils.profiler import PipelineProfiler
from utils.send_mail import *

In [3]:
base = BaseLogger()
logger = base.logger
prof = PipelineProfiler(logger)

In [1]:
import os
from pathlib import Path
from glob import glob


print("CWD:", os.getcwd())


ON_CONNECT = os.getenv("RSTUDIO_PRODUCT") == "CONNECT"
BASE_DIR   = "/tmp" if ON_CONNECT else os.path.expanduser("/home/eyzacharis/Energy Bills")
LOG_DIR    = os.path.join(BASE_DIR, "logs")
LOG_FILE   = os.path.join(LOG_DIR, "energy_bills_automation.log")

print("LOG_DIR :", LOG_DIR)
print("LOG_FILE:", LOG_FILE)
print("LOG_FILE exists:", Path(LOG_FILE).exists())


for search_root in ["/tmp", "/opt/rstudio-connect/mnt", os.getcwd()]:
    matches = glob(f"{search_root}/**/*.log", recursive=True)
    if matches:
        print(f"\nFound in {search_root}:")
        for m in matches:
            print(f"  {m}")
    else:
        print(f"\nNo .log files under {search_root}")

CWD: /home/eyzacharis/Energy Bills
LOG_DIR : /home/eyzacharis/Energy Bills/logs
LOG_FILE: /home/eyzacharis/Energy Bills/logs/energy_bills_automation.log
LOG_FILE exists: True

No .log files under /tmp

No .log files under /opt/rstudio-connect/mnt

Found in /home/eyzacharis/Energy Bills:
  /home/eyzacharis/Energy Bills/logs/Energy_bills.log
  /home/eyzacharis/Energy Bills/logs/energy_bills_automation.log


In [ ]:
import sys, vertica_python, pandas, numpy, os, socket
if load_dotenv(dotenv_path=f"{Path(os.getenv('HOME'))}/dev.env",  override=True):
    conn_info = dict(dotenv_values(f"{Path(os.getenv('HOME'))}/dev.env"))
    logger.info("Loaded credentials from environment file")
else:
    logger.warning("Failed to get credentials from environment file. Reverting to default [if exist]")
    logger.info("Available .env files:\n"+'\n'.join([f for f in os.listdir(os.getenv('HOME')) if '.env' in f]))
    load_dotenv()
    conn_info = dict(dotenv_values())

print("Python     :", sys.version)
print("vertica-py :", vertica_python.__version__)
print("pandas     :", pandas.__version__)
print("numpy      :", numpy.__version__)

nodes = ["energy-up01", "energy-up02", "energy-up03"]
port  = int(os.getenv("port", 5433))
for node in nodes:
    try:
        s = socket.create_connection((node, port), timeout=5)
        s.close()
        print(f" {node}:{port} reachable")
    except Exception as e:
        print(f" {node}:{port} BLOCKED — {e}")

test_file = os.path.join(LOG_DIR, ".write_test")
try:
    with open(test_file, "w") as f:
        f.write("ok")
    os.remove(test_file)
    print(f" LOG_DIR writable: {LOG_DIR}")
except Exception as e:
    print(f" LOG_DIR NOT writable: {e}")
    
with vertica_python.connect(**conn_info) as conn:
    cur = conn.cursor()
    cur.execute("SELECT node_name, user_name, client_hostname FROM sessions WHERE session_id = current_session()")
    row = cur.fetchone()
    print(f" Connected to node: {row[0]} | user: {row[1]} | from: {row[2]}")  
    
env_path = Path(os.getenv("HOME")) / "dev.env"
if env_path.exists():
    print(f" .env found: {env_path}")
else:
    print(f" .env NOT found at {env_path}")
    # list what .env-like files do exist
    found = [f for f in os.listdir(os.getenv("HOME")) if ".env" in f]
    print(f"  Available: {found}")
    
print(f"ON_CONNECT       : {os.getenv('RSTUDIO_PRODUCT') == 'CONNECT'}")
print(f"RSTUDIO_PRODUCT  : {os.getenv('RSTUDIO_PRODUCT')}")
print(f"BASE_DIR resolved: {BASE_DIR}")

In [4]:
# if load_dotenv(dotenv_path=f"{Path(os.getenv('HOME'))}/dev.env",  override=True):
#     conn_info = dict(dotenv_values(f"{Path(os.getenv('HOME'))}/dev.env"))
#     logger.info("Loaded credentials from environment file")
# else:
#     logger.warning("Failed to get credentials from environment file. Reverting to default [if exist]")
#     logger.info("Available .env files:\n"+'\n'.join([f for f in os.listdir(os.getenv('HOME')) if '.env' in f]))
#     load_dotenv()
#     conn_info = dict(dotenv_values())

# # sanity check
# required = ['host', 'port', 'user', 
#             'password', 'database']
# missing = [key for key in required if key not in conn_info.keys()]
# if missing:
#     logger.error(f"Missing env variables:\n"+'\n'.join(missing))
#     conn_info = {str(re.sub("VERTICA_", "", k)).lower(): v for k, v in conn_info.items()}
#     conn_info['tlsmode'] = 'disable'
#     logger.info("Reverting to default")

24-04-2026 15:57:48 INFO [2136086929.<module>:3]: Loaded credentials from environment file


In [ ]:
with vertica_python.connect(**conn_info) as connection:
    with connection.cursor() as cur:
        logger.info(cur.execute("select count(*) from energy_efficiency.plpl2").fetchall())

In [4]:
# try:
#     df = pd.read_csv(Path(glob("**/*coordinates.csv", recursive=True)[0]))
#     logger.info("Coordinates file has been read successfully")
# except Exception as e:
#     logger.exception(f"Error encountered: {e}")

22-04-2026 15:06:43 INFO [4036028853.<module>:3]: Coordinates file has been read successfully


In [ ]:
# logger.info("Reading table: energy_efficiency.parohes_cosmote")
# with prof.stage("fetch_parohes"):
#     sites = read_vertica_table(conn_info, parohes_cosmote_query)

In [ ]:
# sites.head(20)

In [5]:
# logger.debug("debug message")
# logger.info("info message")
# logger.warning("warning message")
# logger.error("error message")

# try:
#     raise ValueError("test exception")
# except Exception:
#     logger.exception("caught exception test")

22-04-2026 15:06:43 INFO [4096092941.<module>:2]: info message
22-04-2026 15:06:43 WARNING [4096092941.<module>:3]: warning message
22-04-2026 15:06:43 ERROR [4096092941.<module>:4]: error message
22-04-2026 15:06:43 ERROR [4096092941.<module>:9]: caught exception test
Traceback (most recent call last):
  File "/tmp/ipykernel_1649351/4096092941.py", line 7, in <module>
    raise ValueError("test exception")
ValueError: test exception


In [8]:
# # Build final report text and email it
# logger.info(f"{get_log_report(logger=logger)[0]}")
# try:
#     send_report_text(
#         report_text=build_report_text(),
#         subject=f"EnergyA Bills Automation Runtime Report - {datetime.now():%Y-%m-%d %H:%M:%S}",
#         recipients=["e.zacharis@msensis.com"],
#         attachment_paths=[get_log_report(logger=logger)[0]], 
#         logger = logger
#     )
#     logger.info("Final report emailed.")
# except Exception as e:
#     logger.exception(f"ERROR sending final report email: {e}")

20-04-2026 14:39:02 INFO [send_mail.py:140]: Sending email to e.zacharis@msensis.com
20-04-2026 14:39:02 INFO [818290530.py:12]: Final report emailed.
